In [ ]:
# Install and import
!pip install kagglehub catboost -q

import kagglehub
path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")
print("Path:", path)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Task 1: Read data
df = pd.read_csv(os.path.join(path, "Q3_data.csv"))
print(f"Shape: {df.shape}")

In [ ]:
# Task 2: Head
df.head()

In [ ]:
# Task 3: Info
df.info()

In [ ]:
# Task 4: Describe
df.describe()

In [ ]:
# Task 1: Handle missing values - fill with median
print(f"Missing values before: {df.isnull().sum().sum()}")

for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print(f"Missing values after: {df.isnull().sum().sum()}")

In [ ]:
# Task 2: Check duplicates
print(f"Duplicates: {df.duplicated().sum()}")
df = df.drop_duplicates()

In [ ]:
# Task 3: Encode categorical (check if any exist)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")
# No categorical columns in this dataset - all numerical

In [ ]:
# Task 4: Apply StandardScaler (exclude target!)
from sklearn.preprocessing import StandardScaler

target_col = 'Target'
feature_cols = [col for col in df.columns if col != target_col]

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print("Scaling done!")

In [ ]:
# Task 5: Check target imbalance
print("Target distribution:")
print(df[target_col].value_counts())
print(f"\nPercentages:")
print(df[target_col].value_counts(normalize=True) * 100)

ratio = df[target_col].value_counts().max() / df[target_col].value_counts().min()
print(f"\nImbalance ratio: {ratio:.2f}:1")

if ratio > 3:
    print(" IMBALANCED - Use StratifiedKFold and F1-Score")
else:
    print("Relatively balanced")

In [ ]:
import seaborn as sns
# Visualize the distribution
plt.figure(figsize=(8, 5))
ax = sns.countplot(x=target_col, data=df, palette='viridis')
plt.title('Target Variable Distribution (0 = No Default, 1 = Default)')
plt.xlabel('Target Class')
plt.ylabel('Count')

# Add count labels on bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plt.show()

In [ ]:
# Task 1: Split X and y
X = df.drop(target_col, axis=1)
y = df[target_col]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Tasks 2,3,4,5: StratifiedKFold + CatBoost + F1-Score
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []
accuracy_scores = []
final_model = None

print("Training with StratifiedKFold...")
print("=" * 50)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    f1 = f1_score(y_val, y_pred, average='weighted')
    acc = accuracy_score(y_val, y_pred)

    f1_scores.append(f1)
    accuracy_scores.append(acc)
    final_model = model

    print(f"Fold {fold}: F1={f1:.4f}, Acc={acc:.4f}")

print("=" * 50)
print(f"Average F1-Score: {np.mean(f1_scores):.4f}")
print(f"Average Accuracy: {np.mean(accuracy_scores):.4f}")

In [ ]:
# Task 1: Plot feature importance
importances = final_model.feature_importances_
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
importance_df = importance_df.sort_values('Importance', ascending=False)

# Plot top 15
plt.figure(figsize=(10, 8))
top15 = importance_df.head(15)
plt.barh(range(15), top15['Importance'].values[::-1])
plt.yticks(range(15), top15['Feature'].values[::-1])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Identify the golden feature
golden_feature = importance_df.iloc[0]['Feature']
golden_importance = importance_df.iloc[0]['Importance']

print("GOLDEN FEATURE ")
print(f"Name: {golden_feature}")
print(f"Importance: {golden_importance:.4f}")

In [ ]:
# Retrain using only the golden feature
X_golden = X[[golden_feature]]

skf_golden = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_golden = []
acc_golden = []

print(f"Training with only '{golden_feature}'...")

for fold, (train_idx, val_idx) in enumerate(skf_golden.split(X_golden, y), 1):
    X_train_g, X_val_g = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train_g, y_val_g = y.iloc[train_idx], y.iloc[val_idx]

    model_g = CatBoostClassifier(iterations=500, learning_rate=0.1, depth=6, verbose=0, random_state=42)
    model_g.fit(X_train_g, y_train_g)
    y_pred_g = model_g.predict(X_val_g)

    f1_golden.append(f1_score(y_val_g, y_pred_g, average='weighted'))
    acc_golden.append(accuracy_score(y_val_g, y_pred_g))

    print(f"Fold {fold}: F1={f1_golden[-1]:.4f}")

# Compare
print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"Full Model F1:   {np.mean(f1_scores):.4f}")
print(f"Golden Only F1:  {np.mean(f1_golden):.4f}")
print(f"Retention:       {(np.mean(f1_golden)/np.mean(f1_scores))*100:.1f}%")